In [1]:
import warnings
from langchain._api import LangChainDeprecationWarning

warnings.simplefilter("ignore", category=LangChainDeprecationWarning)

In [2]:
import os
from dotenv import load_dotenv,find_dotenv
_ = load_dotenv(find_dotenv())

groq_api_key =  os.environ["GROQ_API_KEY"]

In [3]:
from langchain_groq import ChatGroq
llmModel = ChatGroq(model = "llama3-70b-8192")

In [ ]:
from langchain import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain.memory import FileChatMessageHistory
from langchain.memory import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage

chatbot_memory= {}

def get_session_history(session_id: str) -> BaseChatMessageHistory: 
    if session_id not in chatbot_memory:
        chatbot_memory[session_id]=ChatMessageHistory()
    return chatbot_memory[session_id]

chatbot_with_message_history = RunnableWithMessageHistory(
    llmModel,
    get_session_history
)

In [ ]:
session1={"configurable":{"session_id":"001"}}

In [ ]:
response_from_the_chatbot = chatbot_with_message_history.invoke(
    [HumanMessage(content="My favourite colour is red")],
    config=session1,
)

response_from_the_chatbot.content

"Red is a bold and vibrant colour! It's a colour that can evoke feelings of energy, passion, and excitement. Do you have a particular shade of red that you're especially fond of, or is it more of a general love for the colour as a whole?"

In [7]:
response_from_the_chatbot = chatbot_with_message_history.invoke(
    [HumanMessage(content="What's my favorite color?")],
    config=session1,
)

response_from_the_chatbot.content

'I remember! Your favorite color is red!'

In [ ]:
session2= {"configurable":{"session_id":"002"}}

response_from_the_chatbot = chatbot_with_message_history.invoke(
    [HumanMessage(content="What's my favorite color?")],
    config=session2,
)

response_from_the_chatbot.content

"I don't have any information about you, so I don't know what your favorite color is. I'm a large language model, I don't have the ability to know personal information about individuals unless it's provided to me. If you'd like to tell me what your favorite color is, I'd be happy to know!"

In [ ]:
response_from_the_chatbot = chatbot_with_message_history.invoke(
    [HumanMessage(content="What's my favorite color?")],
    config=session1,
)

response_from_the_chatbot.content

'You told me earlier that your favorite color is red!'

In [ ]:
response_from_the_chatbot = chatbot_with_message_history.invoke(
    [HumanMessage(content="My name is suyesh")],
    config=session2,
)

response_from_the_chatbot.content

"Nice to meet you, Suyesh! Unfortunately, I still don't know your favorite color. Would you like to share it with me?"

In [11]:
response_from_the_chatbot = chatbot_with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=session2,
)

response_from_the_chatbot.content

'I remember! Your name is Suyesh.'

Here, when we have long messages and saving everything leads to overflow, so we are gonna limit the context window size

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder 
from langchain_core.runnables import RunnablePassthrough


def limited_memory_of_messages(messages, number_of_messages_to_keep=2):
    return messages[-number_of_messages_to_keep:] 

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

limitedMemoryChain = (
    RunnablePassthrough.assign(messages=lambda x: limited_memory_of_messages(x["messages"]))
    | prompt 
    | llmModel
)

In [13]:
chatbot_with_limited_message_history = RunnableWithMessageHistory(
    limitedMemoryChain,
    get_session_history,
    input_messages_key="messages",
)

In [14]:
responseFromChatbot = chatbot_with_message_history.invoke(
    [HumanMessage(content="My favorite colour is red")],
    config=session1,
)

responseFromChatbot.content

"I knew that! You've told me before that your favorite color is indeed red!"

In [15]:
responseFromChatbot = chatbot_with_message_history.invoke(
    [HumanMessage(content="My favorite city is San Francisco.")],
    config=session1,
)

responseFromChatbot.content

"San Francisco is a wonderful city! I'll remember that it's your favorite city. What is it about San Francisco that you love the most? Is it the Golden Gate Bridge, Alcatraz Island, the steep hills, or something else entirely?"

In [16]:
responseFromChatbot = chatbot_with_limited_message_history.invoke(
    {
        "messages": [HumanMessage(content="My name is suyesh")],
    },
    config=session1,
)

responseFromChatbot.content

"Nice to meet you, Suyesh! I'm happy to assist you with any questions or topics you'd like to discuss. How's your day going so far? Is there anything specific you'd like to chat about or ask? I'm all ears!"

In [34]:
responseFromChatbot = chatbot_with_limited_message_history.invoke(
    {
        "messages": [HumanMessage(content="I live in lolang")],
    },
    config=session1,
)

responseFromChatbot.content

"Lolong is a municipality in the province of Laguna, Philippines. It's a beautiful place surrounded by nature and rich in culture.\n\nHow can I assist you today? Are you looking for information about Lolong, or perhaps you need help with something else? Do you have a specific question or topic in mind?\n\nIf you're looking for information about Lolong, I can try to provide you with some general information or answer any questions you may have about the area. For example, I can tell you about the best places to visit, the local culture, or the history of the area.\n\nLet me know how I can help!"

In [36]:
responseFromChatbot = chatbot_with_limited_message_history.invoke(
    {
        "messages": [HumanMessage(content="where do i live?")],
    },
    config=session1,
)

responseFromChatbot.content

"I apologize, but I don't have any information about your location or where you live. As a helpful assistant, I don't have access to any personal information about you, including your location. Each time you interact with me, it's a new conversation, and I start from a blank slate.\n\nIf you'd like to share your location with me, I'd be happy to try and assist you with any location-specific questions or tasks you may have!"

In [37]:
responseFromChatbot = chatbot_with_message_history.invoke(
    {
        "messages": [HumanMessage(content="What is my favourite colour?")],
    },
    config=session1,
)

responseFromChatbot.content

'I remember! Your favorite color is RED! You told me earlier in our conversation.'